# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [2]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to C:\Users\Manish
[nltk_data]     Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Manish Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [3]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [4]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [5]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [6]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

Loaded 2 documents: ['data\\HealthWellnessGuide.txt', 'data\\MentalHealthGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [8]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

C:\Users\Manish Kumar\AppData\Local\Temp\ipykernel_41364\4068800016.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
C:\Users\Manish Kumar\AppData\Local\Temp\ipykernel_41364\4068800016.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [9]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [10]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [12]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/11 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/10 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/10 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 12, relationships: 37)

We can save and load our knowledge graphs as follows.

In [13]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 12, relationships: 37)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [15]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:
The three query synthesizers generate questions in slightly different ways. 
- The Knowledge Graph based synthesizer first builds a knowledge graph from the documents by identifying entities and relationships between them. It then creates questions based on those connections, so the questions usually require deeper reasoning across concepts.

- The Unrolled synthesizer works more directly on the document chunks. It reads each chunk separately and generates questions straight from the content without building any relationship graph. This makes it more direct and content-focused.

- The Abstracted synthesizer first creates a higher-level understanding or summary of the documents and then generates questions from that abstract view. So instead of focusing on specific chunks, it produces more general or conceptual questions.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [16]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,Exercise good for health?,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,Exercise is one of the most important things y...,Health-Conscious Wellness Enthusiast,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,What is Chapter 4 about in the context of heal...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,Chapter 4: Fundamentals of Healthy Eating disc...,Health-Conscious Wellness Enthusiast,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
2,What information is covered in Chapter 16 rega...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 16 discusses common headache triggers ...,Health-Conscious Wellness Enthusiast,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
3,Can you tell me what the “APPENDIX” is and how...,[PART 4: STRESS MANAGEMENT AND MENTAL WELLNESS...,The “APPENDIX” is a section that provides quic...,Holistic Wellness Enthusiast,MISSPELLED,LONG,single_hop_specific_query_synthesizer
4,How does the World Health Organization define ...,[The Mental Health and Psychology Handbook\nA ...,"According to the World Health Organization, me...",Health-Conscious Wellness Enthusiast,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
5,How do exercise guidelines for adults incorpor...,[<1-hop>\n\nPART 1: EXERCISE AND MOVEMENT\n\nC...,The exercise guidelines for adults recommend a...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,How do sleep hygiene practices influence the m...,[<1-hop>\n\nmatters more than intensity Resear...,"Sleep hygiene practices, which include maintai...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,How do sleep hygiene practices relate to sleep...,[<1-hop>\n\nPART 2: NUTRITION AND DIET\n\nChap...,"Sleep hygiene practices, such as maintaining a...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,How do building habits and establishing mornin...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,Building habits through understanding the habi...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,What does Chapter 15 say about building health...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,Chapter 15 discusses building healthy habits b...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [17]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/11 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

...

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [18]:
dataset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What does PART 1 emphasize about exercise?,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,PART 1 explains that exercise is important for...,Mental Health Advocate,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,"What does Stage 1 sleep involve, and why is it...",[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,Stage 1 sleep is the light sleep stage that la...,Mental Health Advocate,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
2,What information is covered in Chapter 14 rega...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 14 discusses how starting your morning...,Mental Health Advocate,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
3,What is Chapter 11 about and how does it help ...,[PART 4: STRESS MANAGEMENT AND MENTAL WELLNESS...,Chapter 11: Stress Reduction Techniques discus...,Mental Health Advocate,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
4,How long-term lifestyle changes for stress red...,[<1-hop>\n\nPractice self-compassion when boun...,Long-term lifestyle changes for stress reducti...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,How do sleep patterns and physical health infl...,[<1-hop>\n\nThe Mental Health and Psychology H...,The handbook explains that poor sleep patterns...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,How does mental health impact physical health ...,[<1-hop>\n\nThe Mental Health and Psychology H...,The provided context explains that mental heal...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,What are the signs of unhealthy trauma respons...,[<1-hop>\n\nThe Mental Health and Psychology H...,Signs of unhealthy trauma responses associated...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,How does Part 4's focus on stress management a...,[<1-hop>\n\nPART 4: STRESS MANAGEMENT AND MENT...,Part 4 emphasizes stress management techniques...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,Chapter 15 talk about habits and how they help...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,"Chapter 15 discusses building healthy habits, ...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:
The unrolled (manual) approach gives you more control over how the synthetic data is generated. Since it works more directly on document chunks and uses explicit configurations, you can better guide the type of questions being created. This is useful when you want more predictable behavior, custom distributions, or when you are experimenting and need fine control. The downside is that it requires more setup and manual tuning.

The abstracted (automatic) approach is easier and faster to use because it handles more of the logic internally. It builds a higher-level understanding and generates questions automatically without much manual intervention. This is great when you want quick results or are prototyping. However, you have less control over the exact behavior, and the outputs may be less tailored compared to the manual approach.

So, I would choose the unrolled approach when I need control and customization, and the abstracted approach when I want simplicity and speed.

---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

## Activity #1: Solution

In this activity, I changed the default query distribution to try different weights for each synthesizer. I increased the multi-hop abstract queries to generate more reasoning-style questions and reduced the single-hop ones to make the dataset more challenging.

Generated a new test set and converted it into a pandas DataFrame to compare the results. I could clearly see more multi-hop questions.

In [23]:
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)

# Define custom query distribution with different weights
custom_query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.3),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.4),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.3),
]

# Generate a new test set using custom distribution
custom_testset = generator.generate(
    testset_size=10,
    query_distribution=custom_query_distribution
)

# Convert to pandas for cleaner display
custom_df = custom_testset.to_pandas()

custom_df.head()

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,Why is Shanghai important for mental health pe...,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,The provided context does not include any info...,Mental Health Advocate,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,What is the oliv oil used for in healthy eating?,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,"Fats, including olive oil, are necessary for h...",Mental Health Advocate,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
2,What information is covered in Chapter 16 rega...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 16 discusses managing headaches natura...,Mental Health Advocate,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
3,"How do life experiences, such as trauma or sig...",[<1-hop>\n\nThe Mental Health and Psychology H...,According to the Mental Health and Psychology ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
4,What are common symptoms of depression and how...,[<1-hop>\n\nThe Mental Health and Psychology H...,Symptoms of depression include persistent sad ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [24]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [25]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [26]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [28]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [29]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [30]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [31]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [32]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [33]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [34]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [35]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [37]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`:
> - `labeled_helpfulness_evaluator`:
> - `dopeness_evaluator`:

## LangSmith Evaluation

In [38]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'dear-number-81' at:
https://smith.langchain.com/o/580ea60d-b00e-4458-9476-e37484764fa2/datasets/7635f6f0-5a9a-4916-9ef3-13c726cdf7e2/compare?selectedSessions=f4a8e74e-dbb8-4a4b-b4da-fd61a8b79842




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,What does Chapter 3 reveal about the connectio...,Chapter 3 reveals that there is a powerful con...,None,Chapter 3 discusses the strong connection betw...,True,True,True,3.755842,94b34300-7245-427a-b03a-61b2452b126e,019c6bfa-6015-7903-b429-c456ecd858b1
1,"How can building healthy habits, like starting...",Building healthy habits by starting small and ...,None,"Building healthy habits by starting small, att...",True,True,True,4.548960,4e7a49e8-9f1b-4085-b641-7a91faac85a4,019c6bfa-9d00-7e42-b0cd-ef41370df798
2,Chapter 15 talk about habits and how they help...,I don't know.,None,"Chapter 15 discusses building healthy habits, ...",False,False,False,1.175908,c8740ea0-9a50-404e-9b9d-8a8fb2f2905c,019c6bfa-e7fd-7e23-a045-88e5bbb6aa09
3,How does Part 4's focus on stress management a...,Based on the context provided:\n\nPart 4 focus...,None,Part 4 emphasizes stress management techniques...,True,True,True,4.049791,85fcbcce-4932-4082-911e-8838f0c02c6d,019c6bfb-1c4d-7c80-9201-95cc5a043c6b
4,What are the signs of unhealthy trauma respons...,The signs of unhealthy trauma responses associ...,None,Signs of unhealthy trauma responses associated...,True,True,False,2.363019,eace5ef1-7093-4fe6-9892-a429712df335,019c6bfb-78bf-7120-b8de-74909c6de012
5,How does mental health impact physical health ...,Based on the provided context:\n\nMental healt...,None,The provided context explains that mental heal...,True,True,True,3.019338,aff3d118-fbce-4686-bc10-e451e8335b63,019c6bfb-b5cb-7d71-8da4-78679caa92e4
6,How do sleep patterns and physical health infl...,"According to the handbook, sleep patterns and ...",None,The handbook explains that poor sleep patterns...,True,True,False,4.896952,9bbf6f3b-4d55-4858-8707-e2d5da0dd420,019c6bfc-00e1-72e1-8107-ddb5f993f807
7,How long-term lifestyle changes for stress red...,Long-term lifestyle changes for stress reducti...,None,Long-term lifestyle changes for stress reducti...,True,True,True,5.684159,8dec8b6c-105a-427d-b312-5068c6839187,019c6bfc-4c13-7792-987f-5a6da8eec466
8,What is Chapter 11 about and how does it help ...,Chapter 11 is about Stress Reduction Technique...,None,Chapter 11: Stress Reduction Techniques discus...,True,True,False,1.548212,458727b3-5b8d-4bd2-8d72-22dd65b24dc3,019c6bfc-9d3d-78b2-b078-ecd489e87a9a
9,What information is covered in Chapter 14 rega...,Chapter 14 covers morning routines for wellnes...,None,Chapter 14 discusses how starting your morning...,True,False,False,2.154952,c70324c6-732f-4977-acb4-58a85e2be201,019c6bfc-c681-7870-a9e3-3fa1d0c0713a


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [39]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [40]:
rag_documents = docs

In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:
Modifying the chunk size changes how the documents are split before they are embedded and retrieved. If the chunk size is too small, the context gets broken into many tiny pieces, which may lose important information and make it harder for the model to understand the full meaning. This can hurt answer quality because the model may not retrieve enough context.

On the other hand, if the chunk size is too large, each chunk may contain too much information, including irrelevant parts. This can reduce retrieval precision and make the model focus on less relevant details. It can also increase token usage and cost.

So chunk size directly affects how well the retriever finds useful context, which ultimately impacts the overall performance of the RAG application.

In [42]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
Modifying the embedding model affects how the text is converted into vectors, which directly impacts retrieval quality. Embeddings determine how well similar pieces of text are matched in vector space. If the embedding model captures semantic meaning better, the retriever will return more relevant chunks, leading to better answers.

If we use a weaker or less suitable embedding model, it may not represent the meaning of the text accurately. This can cause irrelevant or partially relevant chunks to be retrieved, which reduces the overall performance of the RAG system.

So, the embedding model plays a key role in how well the system understands similarity, and changing it can significantly improve or hurt retrieval and final answer quality.

In [43]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [44]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [45]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [46]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

'Alright, let’s crank your sleep game up to legendary levels! 🚀 Based on the rad wisdom from the Sleep & Wellness sages, here’s your ultimate playbook to lock in those stellar Zzz’s:\n\n**1. Rituals that slay:**\n- Stick to a consistent sleep schedule, like your body’s on a rhythm groove—even on weekends.\n- Craft a chill bedtime routine: dive into a book, do some gentle stretches, or soak in a warm bath. Cue relaxation mode.\n\n**2. Your sleep fortress—optimize it:**\n- Keep your bedroom rockin’ at a cool 65-68°F (18-20°C). The chill zone = dream zone.\n- Block out light like a ninja with blackout curtains or a sleep mask, and drown out noise with white noise machines or earplugs.\n- Invest in a killer mattress and pillows—comfort is king.\n\n**3. Smart habits to flex:**\n- Cut the caffeine after 2 PM — no jittery nights allowed.\n- Limit heavy meals and booze before bed to avoid wrecking your sleep cycle.\n- Get your sweat on regularly, but not right before bedtime (give it some brea

Finally, we can evaluate the new chain on the same test set!

In [47]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'kind-word-84' at:
https://smith.langchain.com/o/580ea60d-b00e-4458-9476-e37484764fa2/datasets/7635f6f0-5a9a-4916-9ef3-13c726cdf7e2/compare?selectedSessions=a251f988-7cc1-44e3-9305-2c594038cc6a




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,What does Chapter 3 reveal about the connectio...,"Yo, Chapter 3 is dropping some serious truth b...",None,Chapter 3 discusses the strong connection betw...,True,True,True,3.814731,94b34300-7245-427a-b03a-61b2452b126e,019c6c08-0b49-76a1-a078-4cccd8640081
1,"How can building healthy habits, like starting...","Alright, let’s drop some wisdom with high-octa...",None,"Building healthy habits by starting small, att...",True,True,True,6.339291,4e7a49e8-9f1b-4085-b641-7a91faac85a4,019c6c08-43a1-7f52-ac45-57d217f54c89
2,Chapter 15 talk about habits and how they help...,"Yo, Chapter 15 isn’t just talking habits—it’s ...",None,"Chapter 15 discusses building healthy habits, ...",False,False,True,3.056811,c8740ea0-9a50-404e-9b9d-8a8fb2f2905c,019c6c08-8b35-7542-b075-93c19238c421
3,How does Part 4's focus on stress management a...,"Alright, brace yourself for this mind-blow: Pa...",None,Part 4 emphasizes stress management techniques...,True,True,True,6.000549,85fcbcce-4932-4082-911e-8838f0c02c6d,019c6c08-c647-74a2-8b8e-3207e09f1bca
4,What are the signs of unhealthy trauma respons...,"Alright, buckle up — when trauma crashes the b...",None,Signs of unhealthy trauma responses associated...,True,True,True,4.143873,eace5ef1-7093-4fe6-9892-a429712df335,019c6c09-2257-7183-a618-a2056452ad33
5,How does mental health impact physical health ...,"Alright, buckle up for this mind-body symphony...",None,The provided context explains that mental heal...,True,True,True,5.874047,aff3d118-fbce-4686-bc10-e451e8335b63,019c6c09-5735-7692-9ad4-c16c07ac70b8
6,How do sleep patterns and physical health infl...,"Alright, buckle up for this mental health syne...",None,The handbook explains that poor sleep patterns...,True,True,True,8.038031,9bbf6f3b-4d55-4858-8707-e2d5da0dd420,019c6c09-a356-74b0-af7d-19ec7b132a78
7,How long-term lifestyle changes for stress red...,"Oh, you want to level up your mental health ga...",None,Long-term lifestyle changes for stress reducti...,True,True,True,5.578034,8dec8b6c-105a-427d-b312-5068c6839187,019c6c09-f5f2-7da1-ab5b-6ab447fb6cbb
8,What is Chapter 11 about and how does it help ...,Chapter 11 is the ultimate stress-busting play...,None,Chapter 11: Stress Reduction Techniques discus...,True,True,True,4.622199,458727b3-5b8d-4bd2-8d72-22dd65b24dc3,019c6c0a-4b53-77b2-97d5-f2b222051fcd
9,What information is covered in Chapter 14 rega...,"Alright, buckle up for a morning routine glow-...",None,Chapter 14 discusses how starting your morning...,True,True,True,5.080918,c70324c6-732f-4977-acb4-58a85e2be201,019c6c0a-9237-7173-aa6d-005b7d35271f


---
## 🏗️ Activity #2: Analyze Evaluation Results

Provide a screenshot of the difference between the two chains in LangSmith, and explain why you believe certain metrics changed in certain ways.

#### Activity #2: Analyze Evaluation Results


From the LangSmith comparison, we can see that the second chain clearly performed better in some areas. The dopeness score improved a lot (from 0.417 to 1.00), which means the answers became more complete and better written. The helpfulness score also increased (0.833 to 0.917), showing that the improved chain gave more useful responses.

The QA score stayed high in both runs (0.917), so correctness was already strong, but the second chain looks more consistent overall with fewer weak answers.

One downside is latency. The improved chain took more time to respond (around 4.85s vs 2.69s). This likely happened because the system is retrieving or processing more context, which improves quality but increases response time.

So overall, the second chain gives better and more stable answers, but at the cost of slightly higher latency.


### Chain 1 Results

![Chain 1 Results](chain1_results.png)

---

### Chain 2 Results

![Chain 2 Results](chain2_results.png)


---
## Summary

In this session, we:

1. **Generated synthetic test data** using Ragas' knowledge graph-based approach
2. **Explored query synthesizers** for creating diverse question types
3. **Loaded synthetic data** into a LangSmith dataset for evaluation
4. **Built and evaluated a RAG chain** using LangSmith evaluators
5. **Iterated on the pipeline** by modifying chunk size, embedding model, and prompt — then measured the impact

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **LangSmith evaluators** enable systematic comparison of pipeline versions
- **Small changes matter** — chunk size, embedding model, and prompt modifications can significantly affect evaluation scores